<img src="../data/zls.svg" width="120" style="float:right; margin: 0 20px 10px 20px;">

# 05 — Advanced: Build a Small Client

> **This notebook is optional / stretch material.**

Goal: wrap the API in a small Python class so all the calls live in one place.

### The client class

Read through this class before running it. A few things to notice:
- All methods use `self.session_id` so you don't have to pass it around
- `poll` has a `max_polls` limit to avoid waiting forever
- `stt` converts the boolean to a lowercase string — the API expects `"true"` / `"false"`, not Python's `True` / `False`

In [ ]:
import base64
import requests
import time
from pathlib import Path


class SproochmaschinnClient:
    def __init__(self, base_url="https://sproochmaschinn.lu"):
        self.base_url = base_url
        self.session_id = None

    def create_session(self):
        data = requests.post(f"{self.base_url}/api/session").json()
        self.session_id = data["session_id"]
        return self.session_id

    def tts(self, text, model="claude"):
        data = requests.post(
            f"{self.base_url}/api/tts/{self.session_id}",
            json={"text": text, "model": model}
        ).json()
        return data["request_id"]

    def stt(self, audio_path, enable_speaker_identification=True):
        with open(audio_path, "rb") as f:
            data = requests.post(
                f"{self.base_url}/api/stt/{self.session_id}",
                files={"audio": f},
                data={
                    "enable_speaker_identification": str(enable_speaker_identification).lower()
                }
            ).json()
        return data["request_id"]

    def poll(self, request_id, sleep_seconds=1, max_polls=120):
        for _ in range(max_polls):
            result = requests.get(f"{self.base_url}/api/result/{request_id}").json()
            if result["status"] == "completed":
                return result
            if result["status"] == "failed":
                raise RuntimeError(f"Request failed: {result}")
            time.sleep(sleep_seconds)
        raise TimeoutError(f"Request {request_id} did not complete in time.")

### Try it out

Create a client instance and open a session.

In [ ]:
client = SproochmaschinnClient()
client.create_session()
print("Session:", client.session_id)

Submit a TTS request and poll for the result — both in one cell since the client keeps things concise.

In [ ]:
request_id = client.tts("Lëtzebuerg ass e schéint Land!")
result = client.poll(request_id)
result

Decode the audio from the result and save it as a WAV file.

In [ ]:
audio_bytes = base64.b64decode(result["result"]["data"])
Path("client_output.wav").write_bytes(audio_bytes)
print("Saved client_output.wav")

Listen to the generated audio.

In [ ]:
from IPython.display import Audio

Audio("client_output.wav")

## Things to think about

- What happens if you call `client.tts(...)` *before* `client.create_session()`? Try it and read the error.
- What would you add to make this class production-ready? (Think: error handling, retries, logging)

## Extension ideas

1. Add an `export()` method that wraps the export endpoint
2. Add a method that does TTS and automatically saves the WAV in one call
3. Add a method that returns STT words as a pandas DataFrame

In [ ]:
# Start here
